Suppose that

$$
\begin{align}

z & \sim \mathrm{normal}(0, 2) \\
x & \sim \mathrm{normal}(z, 6),

\end{align}
$$

where the second argument to $\mathrm{normal}$ is the scale / standard deviation. Then What is

$$\mathbb{E}[z \vert y=-10]?$$

The exact answer is $-1$. But this notebook demonstrates how to estimate the answer using pangolin, JAGS, PyMC, Pyro, NumPyro, JAGS, and Stan.

In [23]:
# setup
import numpy as np

In [25]:
# pangolin

import pangolin
from pangolin import interface as pi

z = pi.normal(0,2)
x = pi.normal(z,6)

E_z = pangolin.blackjax.E(z,x,-10)

print(E_z)

-1.0569503


In [ ]:
# pymc

import pymc as pm

with pm.Model():
    z = pm.Normal('z', 0, 2)
    x = pm.Normal('x', z, 6, observed=-10)
    trace = pm.sample(chains=1)
    z_samps = trace.posterior['z'].values
    E_z = np.mean(z_samps)

print(E_z)

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [z]


Output()

Sampling 1 chain for 1_000 tune and 1_000 draw iterations (1_000 + 1_000 draws total) took 0 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks


-1.076358727495944


In [ ]:
# pyro

import pyro
import torch

def model():
    z = pyro.sample('z', pyro.distributions.Normal(0, 2))
    x = pyro.sample('x', pyro.distributions.Normal(z, 6), obs=torch.tensor(-10.0))

nuts_kernel = pyro.infer.mcmc.NUTS(model)
mcmc = pyro.infer.mcmc.MCMC(nuts_kernel, warmup_steps=500, num_samples=1000, num_chains=1)
mcmc.run()
z_samps = mcmc.get_samples()['z'].numpy()
E_z = np.mean(z_samps)

print(E_z)


Sample: 100%|██████████| 1500/1500 [00:01, 902.31it/s, step size=1.07e+00, acc. prob=0.899] 

-1.0161589


In [40]:
# numpyro

import numpyro
import jax
import jax.numpy as jnp

def model():
    z = numpyro.sample('z', numpyro.distributions.Normal(0, 2))
    x = numpyro.sample('x', numpyro.distributions.Normal(z, 6), obs=-10)

nuts_kernel = numpyro.infer.NUTS(model)
mcmc = numpyro.infer.MCMC(nuts_kernel, num_warmup=500, num_samples=1000, num_chains=1)
mcmc.run(jax.random.PRNGKey(42))
z_samps = mcmc.get_samples()['z']
E_z = np.mean(z_samps)

print(E_z)

sample: 100%|██████████| 1500/1500 [00:00<00:00, 1999.43it/s, 1 steps of size 1.03e+00. acc. prob=0.91]

-1.1216512


In [42]:
# pyjags

import pyjags

model_code = """
model {
  z ~ dnorm(0, 1/2^2)
  x ~ dnorm(z, 1/6^2)
}
"""

model = pyjags.Model(
    code=model_code,
    data={'x': -10},
    chains=1,
    adapt=500
)

samples = model.sample(1000, ['z'])
z_samps = samples['z'].flatten()
E_z = np.mean(z_samps)

print(E_z)

sampling: iterations 1000 of 1000, elapsed 0:00:00, remaining 0:00:00
-1.0680263225588738


In [43]:
# stan

import cmdstanpy
import tempfile
from pathlib import Path

stan_code = """
data {
  real x;
}
parameters {
  real z;
}
model {
  z ~ normal(0, 2);
  x ~ normal(z, 6);
}
"""

with tempfile.TemporaryDirectory() as tmpdir:
    stan_file = Path(tmpdir) / "calculator_model.stan"
    stan_file.write_text(stan_code)

    model = cmdstanpy.CmdStanModel(stan_file=str(stan_file))

    fit = model.sample(
        data={'x': -10.0},
        chains=1,
        iter_warmup=500,
        iter_sampling=1000,
        seed=42
    )
    z_samps = fit.stan_variable('z')

    E_z = np.mean(z_samps)

print(E_z)


12:31:53 - cmdstanpy - INFO - compiling stan file /tmp/tmpbstuq2y_/calculator_model.stan to exe file /tmp/tmpbstuq2y_/calculator_model
12:31:59 - cmdstanpy - INFO - compiled model executable: /tmp/tmpbstuq2y_/calculator_model
12:31:59 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1500 [00:00<?, ?it/s, (Warmup)]

12:31:59 - cmdstanpy - INFO - CmdStan done processing.



-1.0125009283442
